#### Libraries

In [3]:
# Prevent OpenMP runtime conflict between libraries like PyTorch, TensorFlow, and NumPy on Windows
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from libraries import torch, pd, zipfile

#### Dataset (Test)

In [3]:
# Extract dataset.zip
with zipfile.ZipFile('dataset.zip', 'r') as zip_ref:
    zip_ref.extractall()

In [4]:
# Read Dataset
df = pd.read_csv('./dataset/test.csv')
df.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time
0,75.0,0,582,0,20,1,265000.0,1.9,130,1,0,4
1,90.0,1,47,0,40,1,204000.0,2.1,132,1,1,8
2,62.0,0,231,0,25,1,253000.0,0.9,140,1,1,10
3,87.0,1,149,0,38,0,262000.0,0.9,140,1,0,14
4,80.0,0,148,1,38,0,149000.0,1.9,144,1,1,23


In [5]:
df.describe()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time
count,49.000000,49.000000,49.000000,49.000000,49.000000,49.000000,49.000000,49.000000,49.000000,49.000000,49.000000,49.000000
mean,65.503408,0.530612,664.387755,0.551020,36.408163,0.285714,288415.470000,1.445102,137.122449,0.734694,0.346939,129.285714
std,12.791075,0.504234,1178.830508,0.502545,11.326441,0.456435,128987.868336,0.899695,4.884639,0.446071,0.480929,80.899629
min,45.000000,0.000000,23.000000,0.000000,17.000000,0.000000,75000.000000,0.600000,125.000000,0.000000,0.000000,4.000000
25%,59.000000,0.000000,115.000000,0.000000,25.000000,0.000000,204000.000000,1.000000,134.000000,0.000000,0.000000,67.000000
50%,63.000000,1.000000,280.000000,1.000000,35.000000,0.000000,270000.000000,1.100000,137.000000,1.000000,0.000000,119.000000
75%,75.000000,1.000000,675.000000,1.000000,40.000000,1.000000,350000.000000,1.700000,141.000000,1.000000,1.000000,197.000000
max,90.000000,1.000000,7702.000000,1.000000,60.000000,1.000000,850000.000000,6.100000,145.000000,1.000000,1.000000,280.000000


#### Preprocess

In [6]:
# Convert to tensor
x_test = torch.FloatTensor(df.values)

In [11]:
# Load Mean and Standard Deviation of the train dataset
scaler = torch.load('variables.pt', weights_only=False)
mu = scaler['mu']
std = scaler['std']

# Standardization
x_test = (x_test - mu) / std
x_test.shape

torch.Size([49, 12])

#### Load model

In [5]:
model = torch.load('model.pth', weights_only=False)

#### Prediction

In [18]:
test_preds = []
with torch.no_grad():
    for inputs in x_test:
        test_pred = model(inputs)
        test_preds.append(test_pred.round().item())

        

#### Save predictions

In [19]:
# Add predictions as a new column to the dataframe
df['PREDICTED_DEATH_EVENT'] = test_preds
df.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,PREDICTED_DEATH_EVENT
0,75.0,0,582,0,20,1,265000.0,1.9,130,1,0,4,1.0
1,90.0,1,47,0,40,1,204000.0,2.1,132,1,1,8,1.0
2,62.0,0,231,0,25,1,253000.0,0.9,140,1,1,10,1.0
3,87.0,1,149,0,38,0,262000.0,0.9,140,1,0,14,1.0
4,80.0,0,148,1,38,0,149000.0,1.9,144,1,1,23,1.0


In [20]:
# Save the dataframe on the disk
df.to_csv('./dataset/test_with_predictions.csv', index=False)